In [13]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

In [14]:
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

In [15]:
llm =  ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [16]:
# TOOLS

search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """ Perform a basic arithmetic operation on two numbers."""

    try: 
        if operation == 'add':
            result = first_num + second_num
        elif operation == 'subtract':
            result = first_num - second_num
        elif operation == 'multiply':
            result = first_num * second_num
        elif operation == 'divide':
            result = first_num / second_num
        else:
            return {'error': f"Unsupported operation: {operation}. Supported operations are add, subtract, multiply, divide."}
        return {'first_num': first_num, 'second_num': second_num, 'operation': operation, 'result': result}
    except ZeroDivisionError:
        return {'error': 'Division by zero is not allowed.'}
    except Exception as e:
        return {'error': str(e)}



In [17]:
tools = [search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)

In [18]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call.""" 
    messages = state['messages']